In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.ml.yield_prediction import YieldPredictionModel
from src.utils.config import Config

print("✅ Ready")

## 1. Load and Explore Data

In [ ]:
config = Config()
df = pd.read_parquet(config.data_dir / 'staging' / 'test_results.parquet')

print(f"Data: {len(df):,} records")
print(f"Devices: {df['device_id'].nunique()}")
print(f"Tests per device: {len(df) / df['device_id'].nunique():.0f}")

# Check device-level yield
device_yield = df.groupby('device_id')['result'].apply(
    lambda x: 1 if (x == 'pass').all() else 0
)
print(f"\nDevice yield: {device_yield.mean()*100:.2f}%")
print(f"  Pass: {device_yield.sum()} devices")
print(f"  Fail: {(device_yield == 0).sum()} devices")

## 2. Model Comparison

Train multiple models and compare

In [ ]:
# Train different models
models = {}
model_types = ['logistic', 'random_forest', 'gradient_boosting']

for model_type in model_types:
    print(f"\n{'='*60}")
    print(f"Training {model_type.replace('_', ' ').title()} Model")
    print('='*60)
    
    model = YieldPredictionModel(model_type=model_type)
    metrics = model.train(df, test_size=0.2)
    
    models[model_type] = model
    
    print(f"\nTest Set Performance:")
    print(f"  Accuracy:  {metrics['test']['accuracy']:.4f}")
    print(f"  Precision: {metrics['test']['precision']:.4f}")
    print(f"  Recall:    {metrics['test']['recall']:.4f}")
    print(f"  F1-Score:  {metrics['test']['f1']:.4f}")
    if 'auc_roc' in metrics['test']:
        print(f"  AUC-ROC:   {metrics['test']['auc_roc']:.4f}")
    
    print(f"\nCross-Validation:")
    print(f"  Mean: {metrics['cv_mean']:.4f}")
    print(f"  Std:  {metrics['cv_std']:.4f}")

## 3. Model Performance Comparison

In [ ]:
# Compare metrics
comparison = []
for name, model in models.items():
    comparison.append({
        'Model': name.replace('_', ' ').title(),
        'Accuracy': model.metrics['test']['accuracy'],
        'Precision': model.metrics['test']['precision'],
        'Recall': model.metrics['test']['recall'],
        'F1-Score': model.metrics['test']['f1'],
        'CV Mean': model.metrics['cv_mean']
    })

comparison_df = pd.DataFrame(comparison)
print("\nModel Comparison:")
print(comparison_df.to_string(index=False))

# Visualize
fig = px.bar(
    comparison_df.melt(id_vars='Model', var_name='Metric', value_name='Score'),
    x='Model',
    y='Score',
    color='Metric',
    barmode='group',
    title='Model Performance Comparison',
    text='Score'
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis_range=[0, 1.1])
fig.show()

## 4. Best Model Analysis

Select best model and analyze in detail

In [ ]:
# Select best model (random forest typically performs best)
best_model = models['random_forest']
print("Selected model: Random Forest")
print(f"Test Accuracy: {best_model.metrics['test']['accuracy']:.4f}")

In [ ]:
# Confusion Matrix
fig = best_model.plot_confusion_matrix(df)
fig.show()

In [ ]:
# ROC Curve
fig = best_model.plot_roc_curve(df)
fig.show()

## 5. Feature Importance

In [ ]:
# Get feature importance
importance_df = best_model.feature_importance()
print("\nTop 10 Most Important Features:")
print(importance_df.head(10).to_string(index=False))

# Visualize
fig = best_model.plot_feature_importance(top_n=10)
fig.show()

print("\n🔍 Feature Insights:")
top_3 = importance_df.head(3)
for _, row in top_3.iterrows():
    print(f"  - {row['feature']}: {row['importance']:.4f}")

## 6. Making Predictions

In [ ]:
# Predict on full dataset
predictions = best_model.predict(df)
probabilities = best_model.predict_proba(df)

# Analyze predictions
print("Prediction Summary:")
print(f"  Predicted Pass: {predictions.sum()} devices")
print(f"  Predicted Fail: {(predictions == 0).sum()} devices")

# High-risk devices (low pass probability)
risk_threshold = 0.7
at_risk = probabilities[:, 1] < risk_threshold
print(f"\nHigh-Risk Devices (pass probability < {risk_threshold}):")
print(f"  Count: {at_risk.sum()}")
print(f"  Percentage: {at_risk.sum() / len(predictions) * 100:.1f}%")

## 7. Model Interpretation

In [ ]:
# Analyze what makes devices fail
X, y = best_model.prepare_features(df)

# Compare passing vs failing devices
X_with_target = X.copy()
X_with_target['target'] = y.values

print("\nFeature Comparison: Passing vs Failing Devices")
print("="*60)

for col in X.columns[:5]:  # Top 5 features
    pass_mean = X_with_target[X_with_target['target'] == 1][col].mean()
    fail_mean = X_with_target[X_with_target['target'] == 0][col].mean()
    diff_pct = (fail_mean - pass_mean) / pass_mean * 100 if pass_mean != 0 else 0
    
    print(f"\n{col}:")
    print(f"  Pass: {pass_mean:.2f}")
    print(f"  Fail: {fail_mean:.2f}")
    print(f"  Diff: {diff_pct:+.1f}%")

## 8. Save Model

In [ ]:
# Save best model
model_dir = config.project_root / 'models'
model_dir.mkdir(exist_ok=True)

model_path = model_dir / 'yield_prediction_rf.pkl'
best_model.save_model(model_path)

print(f"✅ Model saved to {model_path}")
print(f"   Size: {model_path.stat().st_size / 1024:.1f} KB")

## 9. Summary

**What we learned:**
- ✅ Feature engineering for ML models
- ✅ Training multiple classification models
- ✅ Model evaluation (accuracy, precision, recall, F1, AUC)
- ✅ Feature importance analysis
- ✅ Model interpretation
- ✅ Making predictions on new data

**Key Findings:**
- Best Model: Random Forest
- Test Accuracy: **{:.2f}%**
- Most Important Features: [see above]
- Model saved for deployment

**Next Steps:**
- Anomaly detection (unsupervised learning)
- Deep learning models
- Deploy model to production dashboard